# Async Transcription with Gladia

## Why async transcription?

Async transcription is a good fit when:
- your audio already exists as a file or URL
- you do **not** need live partial transcripts
- you want a clean backend job flow
- you are processing completed meetings after they end

## What we are building

We'll implement this flow:

`local audio file` → `upload to Gladia` → `create async transcription job` → `poll result` → `print speaker-labeled transcript`

## Step 1: Get your Gladia API key from the Developer Console


In [ ]:
# Adding our API key and specifying the path to the audio file
import os
import requests
import asyncio
import json
import httpx
import mimetypes
from time import sleep
from pathlib import Path

API_BASE = "https://api.gladia.io/v2"

GLADIA_API_KEY = "7c1f......ade"
AUDIO_FILE_PATH = "./Friends snippet.wav"

## Step 2: Upload the audio file

Gladia's `POST /v2/pre-recorded` endpoint expects an **audio URL**, not a local file upload.

So if you're starting from a local recording, the first step is to upload it with `POST /v2/upload`, which returns an `audio_url`.

In [ ]:
import os

file_path = AUDIO_FILE_PATH
_, file_extension = os.path.splitext(file_path)

with open(file_path, "rb") as f:
    file_content = f.read()

headers = {
    "x-gladia-key": GLADIA_API_KEY,
    "accept": "application/json",
}

files = [
    ("audio", (os.path.basename(file_path), file_content, "audio/" + file_extension[1:]))
]

print("- Uploading file to Gladia...")
upload_response = httpx.post(
    url=f"{API_BASE}/upload/",
    headers=headers,
    files=files,
    timeout=120.0,
)

upload_response.raise_for_status()
upload_json = upload_response.json()
upload_json

- Uploading file to Gladia...


{'audio_url': 'https://api.gladia.io/file/fb9c1083-f9dc-424a-adc6-0ba90a47908a',
 'audio_metadata': {'id': 'fb9c1083-f9dc-424a-adc6-0ba90a47908a',
  'filename': 'Friends snippet.wav',
  'extension': 'wav',
  'size': 2496114,
  'audio_duration': 156,
  'number_of_channels': 1}}

## Step 3: Create the async transcription job

Now that the audio file is uploaded, we can send its `audio_url` to Gladia's pre-recorded transcription endpoint.

This is an async workflow:
- we submit the audio for processing
- Gladia returns a transcription job ID
- we poll for the result in the next step

For a meeting assistant, I’ll enable **speaker diarization** so the final transcript shows who said what.

In [ ]:
config = {
    "diarization": True,
    "sentences": True,
}

data = {
    "audio_url": upload_json["audio_url"],
    **config,
}

headers = {
    "x-gladia-key": GLADIA_API_KEY,
    "accept": "application/json",
    "Content-Type": "application/json",
}

print("- Sending transcription request to Gladia...")
post_response = httpx.post(
    url=f"{API_BASE}/pre-recorded",
    headers=headers,
    json=data,
    timeout=120.0,
)

post_response.raise_for_status()
post_json = post_response.json()
post_json

- Sending transcription request to Gladia...


{'id': '8f80a81b-d705-4074-85a5-026353f2d688',
 'result_url': 'https://api.gladia.io/v2/pre-recorded/8f80a81b-d705-4074-85a5-026353f2d688'}

## Step 3: Poll for the result

Because this is an async transcription workflow, the API does not return the final transcript immediately.

Instead, we poll the `result_url` until the transcription job is finished.

*Disclaimer: In a production backend, it's better to use callbacks or webhooks rather than polling.*

In [ ]:
from time import sleep
import json

result_url = post_json.get("result_url")

if not result_url:
    raise ValueError(f"No result_url found in response: {post_json}")

while True:
    print("Polling for results...")
    poll_response = httpx.get(
        url=result_url,
        headers={
            "x-gladia-key": GLADIA_API_KEY,
            "accept": "application/json",
        },
        timeout=120.0,
    )

    poll_response.raise_for_status()
    poll_json = poll_response.json()

    status = poll_json.get("status")
    print("Transcription status:", status)

    if status == "done":
        print("- Transcription done")
        break
    elif status == "error":
        print("- Transcription failed")
        print(json.dumps(poll_json, indent=2, ensure_ascii=False))
        break

    sleep(1)

Polling for results...
Transcription status: done
- Transcription done


In [ ]:
# Let's extract speaker-labeled utterances
utterances = poll_json.get("result", {}).get("transcription", {}).get("utterances", [])

for item in utterances:
    speaker = item.get("speaker", "?")
    text = item.get("text", "").strip()
    if text:
        print(f"Speaker {speaker}: {text}")

Speaker 0: There's nothing to tell.
Speaker 0: It's just some guy I work with.
Speaker 1: Come on.
Speaker 1: You're going out with the guy.
Speaker 1: There's got to be something wrong with him.
Speaker 0: So does he have a hump?
Speaker 0: A hump and a hairpiece?
Speaker 2: Wait,
Speaker 2: does he eat chalk?
Speaker 3: Just because I don't want her to go through what I went through with Carl.
Speaker 3: Okay,
Speaker 2: everybody relax.
Speaker 2: This is not even a date.
Speaker 2: It's just two people going out to dinner and not having sex.
Speaker 0: Sounds like a date to me.
Speaker 2: All right,
Speaker 0: so I'm back.
Speaker 0: Back in high school,
Speaker 0: I'm standing in the middle of the cafeteria,
Speaker 0: and I realize I am totally naked.
Speaker 0: Oh,
Speaker 2: I've had that.
Speaker 0: Then I look down,
Speaker 0: and I realize there is a phone there.
Speaker 1: Instead of...
Speaker 0: That's right.
Speaker 0: All of a sudden,
Speaker 0: the phone starts to ring